# unify_pums.ipynb

This notebook reads in the household and person-level PUMS for a given year, merges them, cleans up the ORIGIN/CHOSEN fields, and injects fields that will be used later on during modeling.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))
from lib import io as lio


In [2]:
sample_size = 1  # proportion of people in PUMS to consider
year = 2018
directory = "us"
base_household_name = "psam_hus"
base_person_name = "psam_pus"

In [3]:
person_cols = set(
    pd.read_csv(f"{directory}/{base_person_name}a_{year}.csv", nrows=0).columns
)
household_cols = set(
    pd.read_csv(f"{directory}/{base_household_name}a_{year}.csv", nrows=0).columns
)

In [4]:
for col in household_cols:
    print(col)

FACCESSP
WGTP32
SRNT
INSP
WGTP16
SERIALNO
TYPE
BLD
FDIALUPP
WGTP64
FBLDP
FSTOVP
WKEXREL
WGTP12
TAXAMT
RNTM
FFULP
FTAXP
HUGCL
DIVISION
MHP
LNGI
WGTP3
WGTP75
FHFLP
ST
FHISPEEDP
WGTP42
FTELP
MRGP
RT
NPF
WGTP4
RESMODE
FBATHP
TEL
WGTP60
OCPIP
FRWATP
FULFP
FSMXHP
WGTP18
FINCP
WGTP47
RMSP
R65
WGTP63
FS
WGTP36
RWAT
BROADBND
LAPTOP
FBROADBNDP
FOTHSVCEXP
HINCP
FWATP
WGTP72
FULP
FINSP
YBL
WGTP79
WGTP62
HHT
VACS
WGTP39
STOV
WGTP80
WGTP2
WGTP6
WGTP35
FSMP
FACRP
SSMC
GRPIP
WIF
COMPOTHX
FLAPTOPP
WGTP77
FGRNTP
WATP
WGTP45
WGTP34
WGTP71
FGASP
NP
ADJHSG
SVAL
FSMXSP
GASFP
WGTP40
PSF
FMRGXP
WGTP9
FMRGIP
FAGSP
WGTP28
WGTP73
ACR
FRNTP
WGTP49
WGTP8
NPP
AGS
MULTG
SMOCP
WGTP52
WGTP14
GASP
WGTP68
FTENP
FMRGP
FVACSP
FYBLP
NOC
ADJINC
RNTP
FPLMPRP
WGTP20
WGTP19
WGTP37
WGTP69
WATFP
BDSP
FMHP
WGTP21
WGTP51
FHINCP
FSINKP
WGTP
NR
REFR
WGTP27
CONP
WGTP58
WGTP23
WGTP78
MRGI
SATELLITE
FRWATPRP
NRC
FCOMPOTHXP
FCONP
WGTP1
WGTP38
WGTP57
WGTP67
WGTP29
WORKSTAT
HISPEED
FSMOCP
TABLET
FTABLETP
WGTP10
WGTP22
PLMPRP
WGTP76
SINK
H

In [5]:
def filter_person_cols(col: str):
    # weight column
    if col.startswith("PWGTP") and col != "PWGTP":
        return False
    # filter out flag columns
    return col[1:-1] not in person_cols


target_household_cols = {
    "SERIALNO",
    "NP",
    "TYPE",
    "TEN",
    "VALP",
    "VEH",
    "FES",
    "FINCP",
    "FPARC",
    "GRNTP",
    "GRPIP",
    "HHT",
    "HINCP",
    "OCPIP",
    "PARTNER",
    "R18",
    "SMOCP",
    "TAXAMT",
    "WIF",
    "HUPAOC",
    "HUPARC",
    "MULTG",
    "MV",
    "R65",
    "ACR",
    "MRGP",
    "MRGT",
    "NOC",
    "WKEXREL",
    "WORKSTAT",
    "HUGCL",
    "NPF",
    "NPP",
    "NR",
    "NRC",
    # "CPLT",
}


def filter_household_cols(col: str):
    # weight column
    if col.startswith("WGTP"):
        return False
    # flag column
    if col[1:-1] in person_cols:
        return False
    return col in target_household_cols

In [6]:
# reading in the individual PUMS
dfs = []
dfs.append(
    pd.read_csv(
        f"{directory}/{base_person_name}a_{year}.csv", usecols=filter_person_cols
    )
)
dfs.append(
    pd.read_csv(
        f"{directory}/{base_person_name}b_{year}.csv", usecols=filter_person_cols
    )
)
df_p = pd.concat(dfs).reset_index(drop=True)

del dfs

In [7]:
for col in df_p.columns:
    print(col)

RT
SERIALNO
DIVISION
SPORDER
PUMA
REGION
ST
ADJINC
PWGTP
AGEP
CIT
CITWP
COW
DDRS
DEAR
DEYE
DOUT
DPHY
DRAT
DRATX
DREM
ENG
FER
GCL
GCM
GCR
HINS1
HINS2
HINS3
HINS4
HINS5
HINS6
HINS7
INTP
JWMNP
JWRIP
JWTR
LANX
MAR
MARHD
MARHM
MARHT
MARHW
MARHYP
MIG
MIL
MLPA
MLPB
MLPCD
MLPE
MLPFG
MLPH
MLPI
MLPJ
MLPK
NWAB
NWAV
NWLA
NWLK
NWRE
OIP
PAP
RELP
RETP
SCH
SCHG
SCHL
SEMP
SEX
SSIP
SSP
WAGP
WKHP
WKL
WKW
WRK
YOEP
ANC
ANC1P
ANC2P
DECADE
DIS
DRIVESP
ESP
ESR
FOD1P
FOD2P
HICOV
HISP
INDP
JWAP
JWDP
LANP
MIGPUMA
MIGSP
MSP
NAICSP
NATIVITY
NOP
OC
OCCP
PAOC
PERNP
PINCP
POBP
POVPIP
POWPUMA
POWSP
PRIVCOV
PUBCOV
QTRBIR
RAC1P
RAC2P
RAC3P
RACAIAN
RACASN
RACBLK
RACNH
RACNUM
RACPI
RACSOR
RACWHT
RC
SCIENGP
SCIENGRLP
SFN
SFR
VPS
WAOB
FAGEP
FCITWP
FFODP
FHISP
FINDP
FINTP
FJWDP
FJWMNP
FJWRIP
FLANP
FMARHYP
FMIGSP
FMILPP
FMILSP
FOCCP
FOIP
FPAP
FPERNP
FPINCP
FPOBP
FPOWSP
FRACP
FRELP
FRETP
FSEMP
FSSIP
FSSP
FWAGP
FWKHP
FYOEP


In [8]:
# reading in the household PUMS for referencing purposes
dfs = []
dfs.append(
    pd.read_csv(
        f"{directory}/{base_household_name}a_{year}.csv", usecols=filter_household_cols
    )
)
dfs.append(
    pd.read_csv(
        f"{directory}/{base_household_name}b_{year}.csv", usecols=filter_household_cols
    )
)

df_h = pd.concat(dfs).reset_index(drop=True)

del dfs

In [9]:
# cleaning missing identifier data
df_p["MIGPUMA"] = df_p["MIGPUMA"].fillna(0)
df_p["MIGSP"] = df_p["MIGSP"].fillna(0)

In [10]:
# filtering out the PUMS to people 18+ who moved from places in the contiguous united states to other places in the contiguous united states
mask = (
    (df_p["AGEP"] >= 18)
    & (df_p["MIGSP"] <= 56)
    & (~df_p["MIGSP"].isin([2, 15]))  # 2 and 15 correspond to Alaska and Hawaii
    & (df_p["ST"] <= 56)
    & (~df_p["ST"].isin([2, 15]))
    # generally assume that households move as a unit with the reference person as the main representative
    # add 11 (boarders), 12 (housemates/roommates), and 17 (noninstitutionalized GQ)
    & (df_p["RELP"].isin([0, 11, 12, 17]))
)
print(df_p.shape)
df_subset = df_p.loc[mask].copy()
print(df_subset.shape)

(3214539, 159)
(1370688, 159)


In [11]:
# origin is the MIGSP + MIGPUMA
# need ints since it is treated as a float by default
origin = df_subset["MIGSP"].astype(int).astype(str).str.zfill(2) + df_subset[
    "MIGPUMA"
].astype(int).astype(str).str.zfill(5)
# chosen is the current location, ST + PUMA
chosen = df_subset["ST"].astype(int).astype(str).str.zfill(2) + df_subset[
    "PUMA"
].astype(int).astype(str).str.zfill(5)
# people who stayed had their origin is all zeros (due to fillna 0)
df_subset["ORIGIN"] = origin  # migpuma geography
df_subset["CHOSEN"] = chosen  # puma geography

In [12]:
puma_migpuma = lio.load_puma_migpuma("../geometry/equivalencies/puma_migpuma_2010.csv")
puma_migpuma.head()

,State,MIGPUMA
PUMA,,
0100100,01,0100190
0100200,01,0100290
0100301,01,0100290
0100302,01,0100290
0100400,01,0100400


In [13]:
df_subset["ORIGIN"].value_counts()

ORIGIN
0000000    1183497
0603700       5096
1703400       2999
2500390       2917
0800190       2739
            ...   
2201400         33
4806400         32
2202100         30
1302700         30
5401300         27
Name: count, Length: 976, dtype: int64

In [15]:
# backfill the stay origins to the MIGPUMA where they are currently (chosen == origin)
df_subset["ORIGIN"] = np.where(
    df_subset["ORIGIN"] == "0000000",
    puma_migpuma.loc[df_subset["CHOSEN"], "MIGPUMA"],
    df_subset["ORIGIN"],
)
# fill in the origin state with this backfill in place
df_subset["ORIGIN_STATE"] = df_subset["ORIGIN"].str[:2]
df_subset["CHOSEN_MIGPUMA"] = df_subset["CHOSEN"].map(puma_migpuma["MIGPUMA"])

# define STAY as moving outside the MIGPUMA
df_subset["STAY"] = df_subset["CHOSEN_MIGPUMA"] == df_subset["ORIGIN"]

In [16]:
df_subset["ORIGIN"].value_counts()

ORIGIN
0603700    40360
2500390    21916
1703400    18535
0400100    17076
0800190    15733
           ...  
2300600      354
0800400      352
4806900      325
2201600      323
2202100      314
Name: count, Length: 975, dtype: int64

In [17]:
df_subset["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
06    151547
48    107654
12     89363
36     85799
42     57553
17     54732
39     53104
37     45408
26     44076
13     42481
34     37069
51     36793
53     33037
25     31981
47     29399
18     29392
04     28842
29     27527
55     26238
24     25505
08     24594
27     24572
45     22167
01     21153
21     19577
22     19350
41     18771
09     16171
40     15789
19     14453
05     13073
20     12803
28     12642
32     12273
49     11245
31      8592
35      8416
54      8174
16      6823
33      6172
23      6142
44      4790
30      4625
10      4046
46      3874
11      3714
38      3531
50      3091
56      2565
Name: count, dtype: int64

In [18]:
df_subset["CHOSEN"].value_counts()

CHOSEN
0102500    2029
5310200    1936
5500100    1812
5500700    1704
1200500    1438
           ... 
0602904     225
2701402     223
4806807     212
4804634     207
4804633     204
Name: count, Length: 2336, dtype: int64

In [19]:
df_subset["STAY"].value_counts()

STAY
True     1289124
False      81564
Name: count, dtype: int64

In [20]:
# taking sample of PUMS using consistent random seed
df = df_subset.sample(n=round(df_subset.shape[0] * sample_size), random_state=8470897)
df

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,PWGTP,AGEP,...,FSSIP,FSSP,FWAGP,FWKHP,FYOEP,ORIGIN,CHOSEN,ORIGIN_STATE,CHOSEN_MIGPUMA,STAY
769407,P,2018HU0862682,5,1,500,3,12,1013097,50,58,...,0,0,0,0,0,1200500,1200500,12,1200500,True
1763945,P,2018HU1127317,8,1,409,4,32,1013097,56,49,...,0,0,0,0,0,3200400,3200409,32,3200400,True
153485,P,2018HU1320379,7,1,600,3,5,1013097,86,30,...,0,0,0,0,0,0500590,0500600,05,0500590,True
1185511,P,2018HU0627861,4,1,100,2,19,1013097,32,73,...,0,0,0,0,0,1900100,1900100,19,1900100,True
3071659,P,2018HU0264497,9,2,11705,4,53,1013097,71,55,...,0,0,0,0,0,5311700,5311705,53,5311700,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2152900,P,2018HU0856801,5,1,5300,3,37,1013097,54,45,...,0,0,0,0,0,3705390,3705300,37,3705390,True
1227178,P,2018HU1166411,4,1,200,2,20,1013097,67,53,...,0,0,0,0,0,2000200,2000200,20,2000200,True
965218,P,2018HU0491806,8,1,200,4,16,1013097,140,33,...,0,0,0,0,0,1600190,1600200,16,1600190,True
1215714,P,2018HU0587086,4,1,802,2,20,1013097,12,26,...,0,0,0,0,0,2000800,2000802,20,2000800,True


In [21]:
# should not get duplicate measurements for the same SERIALNO
assert df_h["SERIALNO"].value_counts().max() == 1
# merge the household info into df
df = pd.merge(df, df_h, left_on="SERIALNO", right_on="SERIALNO", how="left")

In [22]:
# own child
df["CHILD_UNDER_6"] = np.where(df["HUPAOC"] == 1, 1, 0)
df["CHILD_6_TO_17"] = np.where(df["HUPAOC"] == 2, 1, 0)
df["CHILD"] = np.where(df["HUPAOC"].isin([1, 2, 3]), 1, 0)
df["CHILD"].value_counts()

CHILD
0    1064463
1     306225
Name: count, dtype: int64

In [23]:
df["WORK2_MAR"] = np.where(df["FES"] == 1, 1, 0)
df["WORK1_MAR"] = np.where((df["FES"] <= 4) & (df["FES"] >= 2), 1, 0)
df["SINGLE_PARENT"] = np.where((df["HHT"] == 2) | (df["HHT"] == 3), 1, 0)

In [24]:
df["EDU_NOHIGH"] = np.where(df["SCHL"] <= 15, 1, 0)
df["EDU_HIGH_BUT_NOT_BACHELORS"] = np.where(
    (df["SCHL"] <= 20) & (df["SCHL"] >= 16), 1, 0
)
df["EDU_BACHELORS_OR_HIGHER"] = np.where(df["SCHL"] >= 21, 1, 0)
df["EDU_ONLY_HIGH"] = np.where(df["SCHL"].isin([16, 17]), 1, 0)
df["EDU_SOME_COLLEGE"] = np.where(df["SCHL"].isin([18, 19, 20]), 1, 0)
df["EDU_ONLY_BACHELORS"] = np.where(df["SCHL"] == 21, 1, 0)
df["EDU_GRADUATE_DEG"] = np.where(df["SCHL"] >= 22, 1, 0)
df["EDU_HAS_DEGREE"] = np.where(df["SCHL"] >= 21, 1, 0)
df["EDU_NO_DEGREE"] = np.where(df["SCHL"] <= 20, 1, 0)

In [25]:
df["AGE_UNDER_18"] = np.where(df["AGEP"] < 18, 1, 0)
df["AGE_18_34"] = np.where((df["AGEP"] <= 34) & (df["AGEP"] >= 18), 1, 0)
df["AGE_35_64"] = np.where((df["AGEP"] >= 35) & (df["AGEP"] <= 64), 1, 0)
df["AGE_18_22"] = np.where(df["AGEP"] <= 22, 1, 0)
df["AGE_23_29"] = np.where((df["AGEP"] >= 23) & (df["AGEP"] <= 29), 1, 0)
df["AGE_30_39"] = np.where((df["AGEP"] >= 30) & (df["AGEP"] <= 39), 1, 0)
df["AGE_40_49"] = np.where((df["AGEP"] >= 40) & (df["AGEP"] <= 49), 1, 0)
df["AGE_50_64"] = np.where((df["AGEP"] >= 50) & (df["AGEP"] <= 64), 1, 0)
df["AGE_OVER_65"] = np.where((df["AGEP"] >= 65), 1, 0)
df["FOREIGN"] = np.where(df["NATIVITY"] == 2, 1, 0)

In [26]:
df["IN_COLLEGE"] = np.where((df["SCHG"] == 15) | (df["SCHG"] == 16), 1, 0)

In [27]:
df["WOMAN_WITH_CHILD"] = np.where((df["PAOC"] >= 1) & (df["PAOC"] <= 3), 1, 0)
df["MALE"] = np.where(df["SEX"] == 1, 1, 0)
df["FEMALE"] = np.where(df["SEX"] == 0, 1, 0)

In [28]:
df["MARRIED"] = np.where(df["MAR"] == 1, 1, 0)
df["RECENTLY_WIDOWED_OR_DIVORCED"] = np.where(
    (df["MARHD"] == 1) | (df["MARHW"] == 1), 1, 0
)
df["RECENTLY_MARRIED"] = np.where(df["MARHM"] == 1, 1, 0)
df["MARRIED_MORE_THAN_YEAR"] = np.where(df["MARRIED"] & ~df["RECENTLY_MARRIED"], 1, 0)

In [29]:
df["IN_MILITARY"] = np.where(df["MIL"] == 1, 1, 0)
df["UNEMPLOYED"] = np.where(df["ESR"] == 3, 1, 0)
df["NOT_IN_LABOR_FORCE"] = np.where(df["ESR"] == 6, 1, 0)
df["IN_LABOR_FORCE"] = np.where(df["ESR"] == 6, 0, 1)

In [30]:
df["WHITE"] = np.where(df["RAC1P"] == 1, 1, 0)
df["BLACK"] = np.where(df["RAC1P"] == 2, 1, 0)
df["INDIAN"] = np.where(df["RAC1P"].isin([3, 4, 5]), 1, 0)
df["AAPI"] = np.where(df["RAC1P"].isin([6, 7]), 1, 0)
df["OTHER_RACE"] = np.where(df["RAC1P"].isin([8, 9]), 1, 0)
df["LATINO"] = np.where(df["HISP"] != 1, 1, 0)
# disable other races to unify with the ACS notion of total population not hispanic of latino american
for col in ["WHITE", "BLACK", "INDIAN", "AAPI", "OTHER_RACE"]:
    df[col] = np.where(df["LATINO"] == 1, 0, df[col])

# make another column making latino a category among races
df["RACE_ETHNICITY"] = np.where(df["LATINO"] == 1, 99, df["RAC1P"])

In [31]:
df.shape

(1370688, 242)

In [ ]:
df.to_parquet(f"pums_{sample_size * 100:.0f}_{year}.parquet", compression="gzip")

In [33]:
# people who've recently had children category
# NOTE: this is a little iffy since this only applies to the women

# def add_recent_child_flag(df: pd.DataFrame) -> pd.DataFrame:
#     """FER_CL cleaning and inference for whether a household recently had a child."""
#     df["FER_CL"] = df["FER"].fillna(0)
#     df["FER_CL"] = np.where(df["FER_CL"] == 2, 0, df["FER_CL"])
#     rec_child = df.groupby("SERIALNO")["FER_CL"].max()
#     df["REC_CHILD"] = rec_child.loc[df["SERIALNO"]].values
#     return df
# df = lclean.add_recent_child_flag(df)
# df["FER_CL"].value_counts()